In [1]:
import numpy as np
import pandas as pd

In [2]:
portfolio = pd.read_csv(
    "../data/processed/portfolio_returns.csv",
    parse_dates=["date"],
)

rolling = pd.read_csv(
    "../data/processed/historical_var_rolling.csv",
    parse_dates=[
        "window_start_date",
        "window_end_date",
        "forecast_date",
        "target_date",
    ],
)

print(portfolio.shape)
print(rolling.shape)

(1637, 6)
(1387, 8)


In [3]:
positions = {
    "first": 0,
    "middle": len(rolling) // 2,
    "last": len(rolling) - 1,
}

rolling.iloc[list(positions.values())][
    [
        "window_start_date",
        "window_end_date",
        "forecast_date",
        "target_date",
        "historical_var",
    ]
]

,window_start_date,window_end_date,forecast_date,target_date,historical_var
0,2020-01-03,2020-12-30,2020-12-30,2020-12-31,0.037728
693,2022-10-13,2023-10-12,2023-10-12,2023-10-13,0.030237
1386,2025-07-25,2026-07-27,2026-07-27,2026-07-28,0.025663


In [4]:
ROLLING_WINDOW = 250
ALPHA = 0.05
TOLERANCE = 1e-12

results = []

for label, pos in positions.items():
    target_index = ROLLING_WINDOW + pos

    window = portfolio.iloc[
        target_index - ROLLING_WINDOW:
        target_index
    ]

    manual_q05 = float(
        np.quantile(
            window["portfolio_simple_return"].to_numpy(),
            ALPHA,
            method="linear",
        )
    )

    manual_var = max(
        0.0,
        -manual_q05,
    )

    a_var = float(
        rolling.iloc[pos]["historical_var"]
    )

    difference = abs(
        manual_var - a_var
    )

    results.append(
        {
            "forecast": label,
            "target_date": rolling.iloc[pos]["target_date"],
            "manual_q05": manual_q05,
            "manual_var": manual_var,
            "a_var": a_var,
            "difference": difference,
            "pass": difference <= TOLERANCE,
        }
    )

comparison = pd.DataFrame(results)
comparison

,forecast,target_date,manual_q05,manual_var,a_var,difference,pass
0,first,2020-12-31,-0.037728,0.037728,0.037728,7.632783e-17,True
1,middle,2023-10-13,-0.030237,0.030237,0.030237,6.938894e-17,True
2,last,2026-07-28,-0.025663,0.025663,0.025663,4.163336e-17,True


In [5]:
assert comparison["pass"].all()

print(
    comparison[
        [
            "forecast",
            "target_date",
            "manual_var",
            "a_var",
            "difference",
            "pass",
        ]
    ]
)

  forecast target_date  manual_var     a_var    difference  pass
0    first  2020-12-31    0.037728  0.037728  7.632783e-17  True
1   middle  2023-10-13    0.030237  0.030237  6.938894e-17  True
2     last  2026-07-28    0.025663  0.025663  4.163336e-17  True
